# Spark Declarative Pipelines (DLT): Monitoring & Data Quality

## Overview

This notebook demonstrates:

* 📋 **Declarative ETL** - Define *what* you want, not *how* to compute it
* ✅ **Data Quality Expectations** - Enforce data quality rules
* 📊 **Pipeline Monitoring** - Track pipeline health and data quality metrics
* 🏗️ **DAB Integration** - Infrastructure as code for pipelines

## Key Concepts

### Spark Declarative Pipelines (formerly DLT)

**Declarative approach** - You define tables and their transformations; the framework handles:
* Dependency management (execution order)
* Incremental processing
* Data quality enforcement
* Error handling and retries
* Monitoring and observability

### Data Quality Expectations

**Three enforcement levels:**

1. **`@expect("rule", "condition")`** - Track violations, allow records through
2. **`@expect_or_drop("rule", "condition")`** - Drop invalid records
3. **`@expect_or_fail("rule", "condition")`** - Fail pipeline on any violation

### Pipeline Monitoring

* **Event logs** - Detailed audit trail in `system.event_log`
* **Data quality metrics** - Track expectation pass/fail rates
* **Lineage tracking** - Understand data flow
* **Performance metrics** - Identify bottlenecks

In [0]:
import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *

# DLT is available in pipeline execution context
# This notebook should be added to a Spark Declarative Pipeline
print("✅ DLT library imported")

In [0]:
@dlt.table(
    name="bronze_orders",
    comment="Bronze layer: Raw orders data from cloud storage",
    table_properties={
        "quality": "bronze",
        "pipelines.autoOptimize.managed": "true"
    }
)
def bronze_orders():
    """
    Ingest raw CSV files from cloud storage.
    Bronze tables preserve raw data as-is.
    """
    return (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("header", "true")
            .option("cloudFiles.inferColumnTypes", "true")
            .load("/databricks-datasets/retail-org/customers/")
    )

print("✅ Bronze layer defined")

In [0]:
@dlt.table(
    name="silver_orders",
    comment="Silver layer: Validated and cleansed orders",
    table_properties={
        "quality": "silver"
    }
)
@dlt.expect_or_drop("valid_customer_id", "customer_id IS NOT NULL")
@dlt.expect_or_drop("valid_email", "customer_email RLIKE '^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}$'")
@dlt.expect("customer_city_exists", "customer_city IS NOT NULL")  # Track but allow
@dlt.expect_or_fail("valid_loyalty_segment", "loyalty_segment IN (1, 2, 3, 4, 5)")  # Critical rule
def silver_orders():
    """
    Transform bronze data with quality checks.
    
    Expectations:
    - Drop: Invalid customer IDs or emails
    - Track: Missing cities (for data quality reporting)
    - Fail: Invalid loyalty segments (critical business rule)
    """
    return (
        dlt.read_stream("bronze_orders")
            .withColumn("customer_email", lower(trim(col("customer_email"))))
            .withColumn("customer_city", trim(col("customer_city")))
            .withColumn("processed_at", current_timestamp())
            .select(
                "customer_id",
                "customer_email",
                "customer_name",
                "customer_city",
                "loyalty_segment",
                "processed_at"
            )
    )

print("✅ Silver layer with expectations defined")

In [0]:
@dlt.table(
    name="gold_customer_city_metrics",
    comment="Gold layer: Customer metrics by city",
    table_properties={
        "quality": "gold"
    }
)
def gold_customer_city_metrics():
    """
    Business-ready aggregations for analytics.
    Gold tables are materialized views optimized for BI tools.
    """
    return (
        dlt.read("silver_orders")
            .groupBy("customer_city", "loyalty_segment")
            .agg(
                count("customer_id").alias("customer_count"),
                countDistinct("customer_id").alias("unique_customers")
            )
            .orderBy("customer_city", "loyalty_segment")
    )

print("✅ Gold layer defined")

In [0]:
@dlt.table(
    name="customer_current_state",
    comment="Current state of customers (SCD Type 1 - overwrites)"
)
@dlt.expect_all({
    "valid_customer_id": "customer_id IS NOT NULL",
    "email_format": "customer_email LIKE '%@%'"
})
def customer_current_state():
    """
    Maintains current customer state using SCD Type 1.
    Uses @dlt.expect_all for multiple expectations at once.
    """
    return dlt.read_stream("silver_orders")

print("✅ Streaming table with SCD Type 1 defined")

## 📊 Pipeline Monitoring

### Event Logs

Spark Declarative Pipelines automatically log detailed events to Delta tables in the `system` schema:

**Key event log tables:**

* `system.event_log` - All pipeline events (updates, data quality, errors)
* Dataset-specific tables track metrics per table

### Querying Event Logs

You can query event logs to:

* Track data quality expectation violations
* Monitor pipeline performance
* Debug failures
* Audit data lineage
* Alert on SLA violations

### Event Types

* `flow_definition` - Pipeline structure
* `flow_progress` - Execution progress
* `user_action` - Manual triggers
* `dataset_statistics` - Data quality metrics

In [0]:
%sql
-- Query expectation violations from event logs
-- Replace <pipeline-id> with your actual pipeline ID

SELECT
  timestamp,
  details:flow_definition.output_dataset as dataset,
  details:flow_definition.schema as schema,
  details:flow_progress.metrics.num_output_rows as output_rows,
  details:flow_progress.data_quality.expectations as expectations
FROM event_log(
  table_name => '<your-catalog>.<your-schema>.<table-name>'
)
WHERE event_type = 'flow_progress'
  AND details:flow_progress.data_quality.expectations IS NOT NULL
ORDER BY timestamp DESC
LIMIT 20;

-- View expectation pass/fail rates
SELECT 
  dataset,
  expectation.name,
  expectation.passed_records,
  expectation.failed_records,
  ROUND(expectation.passed_records * 100.0 / 
    (expectation.passed_records + expectation.failed_records), 2) as pass_rate_pct
FROM (
  SELECT 
    details:flow_definition.output_dataset as dataset,
    explode(details:flow_progress.data_quality.expectations) as expectation
  FROM event_log(table_name => '<your-catalog>.<your-schema>.<table-name>')
  WHERE event_type = 'flow_progress'
)
WHERE expectation.failed_records > 0
ORDER BY pass_rate_pct ASC;

In [0]:
%sql
-- Track pipeline execution times and data volumes

SELECT
  date(timestamp) as execution_date,
  details:flow_definition.output_dataset as dataset,
  COUNT(*) as update_count,
  SUM(details:flow_progress.metrics.num_output_rows) as total_rows_processed,
  AVG(details:flow_progress.metrics.num_output_rows) as avg_rows_per_update,
  MAX(details:flow_progress.metrics.num_output_rows) as max_rows_per_update
FROM event_log(table_name => '<your-catalog>.<your-schema>.<table-name>')
WHERE event_type = 'flow_progress'
  AND details:flow_progress.status = 'COMPLETED'
GROUP BY date(timestamp), details:flow_definition.output_dataset
ORDER BY execution_date DESC, dataset;

## 🏗️ Databricks Asset Bundles (DAB)

**DAB** enables **Infrastructure as Code** for Databricks resources including pipelines.

### Project Structure

```
my-dlt-project/
├── databricks.yml          # Bundle configuration
├── src/
│   ├── bronze_layer.py
│   ├── silver_layer.py
│   └── gold_layer.py
└── resources/
    └── pipeline.yml        # Pipeline definition
```

### Benefits

✅ **Version control** - All configs in Git
✅ **Environment promotion** - Dev → Staging → Prod
✅ **Reproducibility** - Consistent deployments
✅ **CI/CD integration** - Automated testing & deployment
✅ **Multi-resource management** - Pipelines, jobs, clusters together

In [0]:
%undefined
# databricks.yml - Main bundle configuration

bundle:
  name: customer-dlt-pipeline

include:
  - resources/*.yml

targets:
  dev:
    mode: development
    default: true
    workspace:
      host: https://your-workspace.cloud.databricks.com
    variables:
      catalog_name: dev_catalog
      pipeline_name: customer_pipeline_dev
  
  prod:
    mode: production
    workspace:
      host: https://your-workspace.cloud.databricks.com
      root_path: /Workspace/Production/Pipelines
    variables:
      catalog_name: prod_catalog
      pipeline_name: customer_pipeline_prod
    run_as:
      service_principal_name: "sp-dlt-pipeline"

In [0]:
%undefined
# resources/pipeline.yml - DLT Pipeline definition

resources:
  pipelines:
    customer_etl_pipeline:
      name: ${var.pipeline_name}
      
      target: ${var.catalog_name}.customer_analytics
      
      libraries:
        - notebook:
            path: ../src/bronze_layer.py
        - notebook:
            path: ../src/silver_layer.py
        - notebook:
            path: ../src/gold_layer.py
      
      clusters:
        - label: default
          autoscale:
            min_workers: 1
            max_workers: 5
            mode: ENHANCED
      
      configuration:
        "pipelines.enableTrackHistory": "true"
        "spark.sql.shuffle.partitions": "auto"
      
      continuous: false
      
      development: ${bundle.target == "dev"}
      
      photon: true
      
      edition: ADVANCED
      
      notifications:
        - email_recipients:
            - data-team@company.com
          alerts:
            - on-update-failure
            - on-update-fatal-failure
            - on-flow-failure

In [0]:
%undefined
# Initialize a new bundle
databricks bundle init

# Validate bundle configuration
databricks bundle validate

# Deploy to dev environment (default)
databricks bundle deploy

# Deploy to production
databricks bundle deploy -t prod

# Run the pipeline
databricks bundle run customer_etl_pipeline

# Destroy resources (careful!)
databricks bundle destroy -t dev

## 🎯 Summary

### Declarative ETL with Spark Declarative Pipelines

✅ **Use `@dlt.table()` for batch** and `@dlt.view()` for intermediate transformations

✅ **Use `@dlt.expect*` decorators** to enforce data quality at transformation time

✅ **Structure as Bronze → Silver → Gold** for clear data quality progression

✅ **Enable Auto Loader** (`cloudFiles`) for incremental ingestion

### Data Quality Best Practices

1. **Bronze layer** - Minimal expectations, preserve raw data
2. **Silver layer** - Enforce critical quality rules with `@expect_or_drop`
3. **Gold layer** - Business logic, assume clean data from Silver
4. **Use `@expect_or_fail`** only for critical violations that should stop the pipeline
5. **Monitor expectation metrics** via event logs

### Monitoring Strategy

* Query `event_log()` function for real-time monitoring
* Set up **alerts** for expectation failures
* Track **data quality trends** over time
* Monitor **pipeline performance** (rows/sec, latency)
* Use **Databricks SQL dashboards** to visualize metrics

### DAB Deployment

1. **Separate environments** - Dev, staging, prod configs
2. **Version control** - All code and configs in Git
3. **CI/CD pipeline** - Automated testing and deployment
4. **Service principals** - For production pipeline execution
5. **Variables & secrets** - Environment-specific configuration

---

## 🚀 Next Steps

1. Create a Spark Declarative Pipeline in the UI
2. Add this notebook to the pipeline
3. Configure source paths and target catalog
4. Run the pipeline and monitor event logs
5. Set up DAB for automated deployment